In [14]:
import os
from langchain_openrouter import ChatOpenRouter
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n", chunk_size=600, chunk_overlap=100
)

loader = UnstructuredFileLoader("./files/document.txt")

docs = loader.load_and_split(text_splitter=splitter)

llm = ChatOpenRouter(
    temperature=0.1,
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

underlying = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
)
store = LocalFileStore("./.cache/embedding")
embedding = CacheBackedEmbeddings.from_bytes_store(
    underlying,
    store,
    namespace=underlying.model,
)
embedding.model = underlying.model

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.runnables import RunnablePassthrough


class Qdrant(QdrantVectorStore):
    pass


client = QdrantClient(":memory:")

client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = Qdrant(
    client=client,
    collection_name="demo_collection",
    embedding=embedding,
)
vector_store.add_documents(docs)

retriver = vector_store.as_retriever()

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "주어진 context만을 이용해 사용자의 질문에 대답하세요. 모르면 모른다고 이야기하세요. 절대로 꾸며내거나 지어내면 안됩니다. \n\n context : {context}",
        ),
        ("human", "{question}"),
    ]
)

chain = {"context": retriver, "question": RunnablePassthrough()} | prompt | llm

chain.invoke("Aaronson 은 유죄인가요?")
chain.invoke("그가 테이블에 어떤 메시지를 썼나요?")
chain.invoke("Julia 는 누구인가요?")

제공된 문맥에 따르면, 윈스턴은 **"Jones, Aaronson, and Rutherford were guilty of the crimes they were charged with."**(존스, 아론슨, 러더포드는 그들에게 제기된 죄목에 대해 유죄였다)라는 생각을 받아들이게 되었다고 나옵니다.

다만, 이 문맥은 윈스턴이 당국의 고문과 세뇌(크라임스톱) 과정을 거쳐 자신의 기억과 진실을 왜곡하여 받아들이게 된 상황임을 보여줍니다. 그는 이전에 그들의 무죄를 증명하는 사진을 본 적이 있다고 기억했지만, 그것은 거짓 기억이며 자신이 꾸며낸 것이라고 스스로를 설득합니다.제공된 문맥에는 그가 테이블에 메시지를 썼다는 내용이 없습니다. 문맥에는 윈스턴이 체스트넛 트리 카페에서 진을 마시고, 체스를 두고, 텔레스크린을 듣거나, 진리부에서 일하는 모습 등이 묘사되어 있지만, 테이블에 메시지를 적는 장면은 나오지 않습니다.제공된 문맥에 따르면, **줄리아(Julia)**는 주인공 **윈스턴(Winston)의 연인이자 반당(反黨) 동지**입니다.

문맥에서 드러나는 그녀에 대한 주요 사실은 다음과 같습니다.

*   **윈스턴과의 관계:** 윈스턴과 비밀리에 연인 관계였으며("그들의 간음, 그들의 모임"), 당에 대항하는 막연한 음모를 꾸미기도 했습니다.
*   **체포와 고문:** 윈스턴과 함께 체포되었으며, 윈스턴은 고문(특히 쥐가 나오는 101호실)에서 그녀를 배신하라고 외쳤습니다("줄리아에게 해! 줄리아에게 해! 나만은 안 돼!").
*   **석방 후 만남:** 석방 후 체스넛 트리 카페에서 윈스턴과 다시 만났으나, 서로 더 이상 예전 같은 감정을 느끼지 못하게 되었습니다("너는 상대방에게 더 이상 예전 같은 감정을 느끼지 않게 돼").
*   **윈스턴의 마음:** 윈스턴은 고문 과정에서 그녀의 습관, 성격, 과거, 만남의 세세한 내용까지 모두 자백했지만, 오브라이언은 윈스턴이 "그녀를 사랑하지 않게 된 것은 아니다"라고 인정했습니다. 윈스턴은 꿈속에서 "줄리아!

AIMessage(content='제공된 문맥에 따르면, **줄리아(Julia)**는 주인공 **윈스턴(Winston)의 연인이자 반당(反黨) 동지**입니다.\n\n문맥에서 드러나는 그녀에 대한 주요 사실은 다음과 같습니다.\n\n*   **윈스턴과의 관계:** 윈스턴과 비밀리에 연인 관계였으며("그들의 간음, 그들의 모임"), 당에 대항하는 막연한 음모를 꾸미기도 했습니다.\n*   **체포와 고문:** 윈스턴과 함께 체포되었으며, 윈스턴은 고문(특히 쥐가 나오는 101호실)에서 그녀를 배신하라고 외쳤습니다("줄리아에게 해! 줄리아에게 해! 나만은 안 돼!").\n*   **석방 후 만남:** 석방 후 체스넛 트리 카페에서 윈스턴과 다시 만났으나, 서로 더 이상 예전 같은 감정을 느끼지 못하게 되었습니다("너는 상대방에게 더 이상 예전 같은 감정을 느끼지 않게 돼").\n*   **윈스턴의 마음:** 윈스턴은 고문 과정에서 그녀의 습관, 성격, 과거, 만남의 세세한 내용까지 모두 자백했지만, 오브라이언은 윈스턴이 "그녀를 사랑하지 않게 된 것은 아니다"라고 인정했습니다. 윈스턴은 꿈속에서 "줄리아! 내 사랑 줄리아!"라고 부르며 그녀를 그리워하기도 했습니다.\n\n요약하자면, 줄리아는 윈스턴이 사랑했고 함께 당에 저항하려 했던 여성이나, 결국 국가의 고문과 세뇌 앞에서 서로를 배신하고 감정마저 잃어버린 비극적인 관계의 상대방입니다.', additional_kwargs={'reasoning_content': 'The user is asking "Who is Julia?" based on the provided context.\nI need to scan the provided documents for information about Julia.\n\nDocument 1:\n- Mentions a conversation between "he" (Winston, implied by context of 1984) and "she" (Julia).\n- "All yo